# Microstate surrogates

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from utils import *
np.set_printoptions(precision=3)

### Statistical analysis

In [ ]:
alpha = 0.05 # significance level
# auxiliary function to print whether test result is NOT/is significant
f_sign = lambda p: 'is NOT' if (p < alpha) else 'is'

### 1. Load a continuous-time microstate sequence
- ms_010002_EC_ep00.txt: Markov-1, shuffled surrogate is compatible with T
- ms_010002_EC_ep01.txt: Markov-1, shuffled surrogate is compatible with T
- ms_010002_EC_ep02.txt: Markov-1, shuffled surrogate is compatible with T
- ms_010003_EC_ep00.txt: NOT Markov-1, shuffled surrogate is NOT compatible with T
- ms_010003_EC_ep01.txt: NOT Markov-1, shuffled surrogate is NOT compatible with T
- ms_010003_EC_ep02.txt: NOT Markov-1, shuffled surrogate is compatible with T
- ms_010006_EC_ep02.txt: Markov-1, shuffled surrogate is NOT compatible with T
- ms_010006_EC_ep06.txt: Markov-1, shuffled surrogate is NOT compatible with T

In [ ]:
data_dir = "./data"
files = os.listdir(data_dir)
files.sort() # make file list reproducible
n_files = len(files)
print(f"Found {n_files:d} files.")
#rnd_idx = np.random.choice(n_files)
rnd_idx = 7
x = np.loadtxt(f"{data_dir:s}/{files[rnd_idx]}").astype(int)
labels = np.unique(x) 
K = len(labels)
print(f"Random file index: {rnd_idx:d}, file: {files[rnd_idx]:s}")
print(f"Continuous microstate sequence, n = {len(x):d} samples.")
print(f"Set of microstate labels: {labels} (K={K:d})")

### 2. Extract the embedded jump sequence

In [ ]:
y = embedded_jump(x)
n = len(y)
print(f"Embedded jump sequence: n={n:d} samples, labels: {np.unique(y)}")

### 3. Markov chain surrogate
- Generate a first-order Markov chain surrogate from the empirical distribution p and the empirical transition matrix T

In [ ]:
p = pmf(y,K) # pmf: probability mass function
T = tpm(y,K) # tpm: (conditional) transition probability matrix
y_mc1 = mc(p,T,n)

### 4. Surrogate from shuffle / duplicate removal
- As in Artoni et al. (2023) https://doi.org/10.1016/j.neuroimage.2023.120196
- Similar to Murphy et al. (2020) https://doi.org/10.1016/j.bpsc.2019.07.006
- Same transition matrix as in Lehmann et al. (2005) https://doi.org/10.1016/j.pscychresns.2004.05.007

In [ ]:
T0 = np.tile(p,[K,1]) # transition matrix of the zero-order Markov chain
T_ = T0 - np.diag(np.diag(T0)) # make zero diagonal (=remove duplicates)
Ts = T_/T_.sum(axis=1, keepdims=True) # re-normalize rows (make row sums = 1) to get shuffling matrix
ps = p_equilibrium(Ts) # get equilibrium distribution of the shuffling matrix Ts (1st eigenvector)

print("First-order Markov chain matrix:\n", T, "\n")
print("Microstate distribution\np   =", p)
print("Check that p = p.T:\np.T =", p@T, "\n")
print("Zero-order Markov chain matrix:\n", T0, "\n")
print("Duplicate removal (zero diagonal):\n", T_, "\n")
print("Shuffling matrix:\n", Ts, "\n")
#print("Microstate distribution after shuffling:", ps, "\n")
#print("Check that ps = ps.Ts:\nps   = ", ps, '\nps.Ts= ', ps@Ts)

# Generate a surrogate from shuffling and duplicate removal using (ps, Ts)
y_sdr = mc(ps,Ts,n)

### 5. Markov tests
- Original jump sequences could be first-order or higher-order
- Both surrogates should be first-order
- No jump sequence (original or jump) can be zero-order

In [ ]:
# Original jump sequence
p0 = test_markov0(y,K)
p1 = test_markov1(y,K)
print(f"The original jump sequence {f_sign(p0):s} Markov-0 (p={p0:.3f})")
print(f"The original jump sequence {f_sign(p1):s} Markov-1 (p={p1:.3f})")

# Markov chain generated surrogate
p0_mc1 = test_markov0(y_mc1,K)
p1_mc1 = test_markov1(y_mc1,K)
print(f"\nThe MC-1 surrogate {f_sign(p0_mc1):s} Markov-0 (p={p0_mc1:.3f})")
print(f"The MC-1 surrogate {f_sign(p1_mc1):s} Markov-1 (p={p1_mc1:.3f})")

# Shuffle/duplicate removal surrogate
p0_sdr = test_markov0(y_sdr,K)
p1_sdr = test_markov1(y_sdr,K)
print(f"\nThe shuffled surrogate {f_sign(p0_sdr):s} Markov-0 (p={p0_sdr:.3f})")
print(f"The shuffled surrogate {f_sign(p1_sdr):s} Markov-1 (p={p1_sdr:.3f})")

### 6. Compare original and surrogate transition matrices

In [ ]:
# Original jump sequence; trivial case, T has been computed from y
p_tm = test_transition_matrix(y, T)
print(f"The original jump sequence {f_sign(p_tm):s} compatible with T (p={p_tm:.3f})")

# Markov chain generated surrogate; should be compatible with T as per construction
p_tm_mc1 = test_transition_matrix(y_mc1, T)
print(f"The MC-1 surrogate {f_sign(p_tm_mc1):s} compatible with T (p={p_tm_mc1:.3f})")

# Shuffle/duplicate removal surrogate; this is the question!
p_tm_sdr = test_transition_matrix(y_sdr, T)
print(f"The shuffled surrogate {f_sign(p_tm_sdr):s} compatible with T (p={p_tm_sdr:.3f})")

## Conclusion
Problems arise when the original jump sequence (`y`) has first-order Markov structure with transition matrix `T`, but is not compatible with the shuffle/duplicate removal method with transition matrix `Ts`.  

In that case, both `y` and the shuffled surrogate `y_sdr` have first-order structure but are classified as different. It would be wrong to conclude that `y` has _more_ syntax than the shuffled  surrogate `y_sdr`.  

This problem does not arise when using the Markov chain surrogate based on the original `T`.